In [ ]:
import random
import numpy as np
import time
import numba
import math
from numpy.random import Generator, PCG64DXSM, SeedSequence
import multiprocessing as mp
from structure_ORIGINAL_FNs_2026_00 import play_sequence

np.set_printoptions(suppress=True)

In [2]:
# use this cell for setting initial perameters

# rng = np.random.default_rng()

sides = 2
pairs = np.asarray([[0, 1], [1, 0], [1, 2], [2, 1], [2, 3], [3, 2], [3, 4], [4, 3]])
testpairs = np.asarray([[1, 3], [3, 1]])
nonadjpairs = np.asarray([[0, 2], [2, 0], [0, 3], [3, 0], [0, 4], [4, 0], [1, 4], [4, 1], [2, 4], [4, 2]])
allpairs = np.concatenate((pairs, nonadjpairs, testpairs))

plen = len(pairs)
alen = len(allpairs)

terms = 5
maxvalue = 10
startstop = 2
noise = 0.
annealing = 0.

timesteps = 10**8
runs = 1000

rein1 = 4
rein2 = 4
punish1 = -11
punish2 = -11
inertia = 1

nsteps = 100
blocklength = nsteps*10


In [3]:
# use this cell for running the code

#measuring how long code takes to run
start = time.perf_counter()



#nprocs = mp.cpu_count()-1
pool = mp.Pool(processes=11)

sg = SeedSequence()
rgs = [Generator(PCG64DXSM(s)) for s in sg.spawn(runs)]




mpseq_final = pool.starmap(play_sequence, [(n, rgs[n], rein1, punish1, rein2, punish2, timesteps, nsteps, sides, pairs, testpairs, nonadjpairs, allpairs, plen, alen, terms, maxvalue, startstop, noise, annealing, runs, inertia, blocklength) for n in range(runs)])

final_sigweights = []
final_cumsuc = []
final_cumsucadd = []
final_testcumsucadd = []
final_recweights = []

for res_idx in range(0, len(mpseq_final)):
    
    final_sigweights.append(mpseq_final[res_idx][0])
    final_cumsuc.append(mpseq_final[res_idx][1])
    final_cumsucadd.append(mpseq_final[res_idx][2])
    final_testcumsucadd.append(mpseq_final[res_idx][3])
    final_recweights.append(mpseq_final[res_idx][4])

final_sigweights = np.asarray(final_sigweights)
final_cumsuc = np.asarray(final_cumsuc)
final_cumsucadd = np.asarray(final_cumsucadd)
final_testcumsucadd = np.asarray(final_testcumsucadd)
final_recweights = np.asarray(final_recweights)



    
print("average cumsuc = ")
print(np.average(final_cumsuc)/runs)
print(" ")

print("average final cumsucadd = ")
print(np.average(final_cumsucadd)/(100))
print(" ")

print("average test cumsum = ")
print(np.average(final_testcumsucadd)/(100))
print(" ")

finish = time.perf_counter()
print(f'Finished in {round(finish-start,0)/60} minutes')

average cumsuc = 
99429.96688600001
 
average final cumsucadd = 
0.9956100000000001
 
average test cumsum = 
0.84595
 
Finished in 513.6 minutes


In [4]:

np.save('structure_noiseAnn_dsktp_ORIGINAL_2s-MV10_sigweights', final_sigweights)
np.save('structure_noiseAnn_dsktp_ORIGINAL_2s-MV10_recweights', final_recweights)
np.save('structure_noiseAnn_dsktp_ORIGINAL_2s-MV10_testcumsucadd', final_testcumsucadd)



In [7]:
cutoff = 90
final_test_count = 0
for cumsum in final_testcumsucadd:
    if cumsum > cutoff:
        final_test_count += 1
print(final_test_count)

635


In [8]:

simple_sig = np.zeros_like(sigweights)
for ax0 in range(0, sides):
    for ax1 in range(0, 2):
        for ax2 in range(0, terms):
            for ax3 in range(0, maxvalue):
                if (sigweights[ax0, ax1, ax2, ax3][0]) > (sigweights[ax0, ax1, ax2, ax3][1]):
                    simple_sig[ax0, ax1, ax2, ax3][0] = 1
                else:
                    simple_sig[ax0, ax1, ax2, ax3][1] = 1

simple_sig = np.sum(simple_sig, axis=3)
print(simple_sig)

for s in range(0, sides):
    for l in range(0, terms):
        for s2 in range(0, sides):
            for r in range(0, terms):
                if l != r:
                    print(f'LEFT side: {s} term: {l} magnitude: {simple_sig[s, 0, l, 1]}')
                    print(f'RIGHT side: {s2} term: {r} magnitude: {simple_sig[s2, 1, r, 1]}')
                    print(f'receiver: {recweights[math.floor(simple_sig[s, 0, l, 1]*(maxvalue+1) + simple_sig[s2, 1, r, 1])]}')

[[[[ 5. 10.]
   [ 6.  9.]
   [ 8.  7.]
   [ 9.  6.]
   [ 7.  8.]]

  [[ 4. 11.]
   [ 9.  6.]
   [ 8.  7.]
   [ 8.  7.]
   [14.  1.]]]


 [[[ 5. 10.]
   [ 6.  9.]
   [ 8.  7.]
   [10.  5.]
   [10.  5.]]

  [[ 4. 11.]
   [ 9.  6.]
   [ 9.  6.]
   [ 8.  7.]
   [15.  0.]]]]
LEFT side: 0 term: 0 magnitude: 10.0
RIGHT side: 0 term: 1 magnitude: 6.0
receiver: [3164154.       1.]
LEFT side: 0 term: 0 magnitude: 10.0
RIGHT side: 0 term: 2 magnitude: 7.0
receiver: [      1. 2921700.]
LEFT side: 0 term: 0 magnitude: 10.0
RIGHT side: 0 term: 3 magnitude: 7.0
receiver: [      1. 2921700.]
LEFT side: 0 term: 0 magnitude: 10.0
RIGHT side: 0 term: 4 magnitude: 1.0
receiver: [    1. 10369.]
LEFT side: 0 term: 0 magnitude: 10.0
RIGHT side: 1 term: 1 magnitude: 6.0
receiver: [3164154.       1.]
LEFT side: 0 term: 0 magnitude: 10.0
RIGHT side: 1 term: 2 magnitude: 6.0
receiver: [3164154.       1.]
LEFT side: 0 term: 0 magnitude: 10.0
RIGHT side: 1 term: 3 magnitude: 7.0
receiver: [      1. 2921700.]
LEFT 

In [13]:
np.set_printoptions(threshold=np.inf)
print(sigweights)

[[[[[ 6964559.        1.]
    [ 6964475.        1.]
    [ 6964503.        1.]
    [ 6964449.        1.]
    [ 6964449.        1.]
    [ 6964523.        1.]
    [       1.  6964523.]
    [ 6964525.        1.]
    [       1.  6964489.]
    [       1.  6964457.]
    [       1.  6964529.]
    [ 6964517.        1.]
    [ 6964489.        1.]
    [       1.  6964469.]
    [ 6964475.        1.]
    [ 6964483.        1.]
    [ 6964457.        1.]
    [       1.  6964465.]
    [       1.  6964485.]
    [       1.  6964469.]]

   [[       7. 12661503.]
    [ 7302905.  5358561.]
    [      13. 12661457.]
    [12661483.        1.]
    [       1. 12661461.]
    [      19. 12661453.]
    [       1. 12661465.]
    [      49. 12661383.]
    [      11. 12661421.]
    [12661461.       13.]
    [      25. 12661439.]
    [12661467.        1.]
    [       1. 12661473.]
    [12661527.        1.]
    [       7. 12661497.]
    [       1. 12661473.]
    [12661499.        1.]
    [12661515.        7.]
    [12661